# Overview of the Action Composer

![full_model_overview](../resources/v0_6/action_composer_overview.png)

The BioJEPA-AC model is designed to support perturbations of different modalities (DNA, Protein, Chemical) with many different modes (crispri, crispra, overexpression, knockout, inhibitor, agonist, degrader, binder). Since these modalities can be of varied lengths and in some cases we might just have a perturbation target, or the perturbation itself, we use an action composer to create a unified representation of the perturbations we may use.

The ActionComposer is a dual-pathway linear encoder that projects sequence and target embeddings into a shared latent space via separate linear projections, fuses them additively, and applies FiLM conditioning from a learned mode embedding to encode the perturbation mechanism. This notebook will walk through each layer so that the reader can build an intuition for what each layer is doing to the data. To this end, you'll see that we set the layer initializations and numbers to ones where you can hand calculate if you need to follow a layer better.

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import math

## Data Prep

We'll start with a simple data prep. We'll use a batch of 2 (representing 2 cells) and we'll support up to 2 perturbation slots.  With the current architecture, we have to predesignate the max perturbations. We'll structure our cells as follows:
1. **Cell 1** receives a CRISPRa perturbation (target protein only, no sequence) and a CRISPRi perturbation (sgRNA sequence + target protein)
2. **Cell 2** receives a chemical inhibitor (SMILES sequence + target protein).  We'll need to pad the second perturbation as you'll see. 

We picked this setup not because it's representative of our training data, but because it demonstrates the breadth of our perturbation support. 

In [2]:
batch = 2
n_perts = 2 

SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

### Data Encoding

In our model, the data prep involves getting the raw sequences (DNA, SMILES, or Amino Acids) and passing them through a foundation model to create an embedding representation of each component. In our example we'll jump straight to mocking up the embeddings.

We'll bring in the dictionaries we use to control indexing. We use a modality-based dictionary to describe the type of data we have and the dimensions it exists in. The mode dictionary represents perturbation impact so you'll see we include learnable mode embeddings as part of the model layers. 

In [3]:
MODALITY_TO_ID = {'dna': 0,'protein': 1,'chemical': 2}
ID_TO_MODALITY = {0: 'dna', 1: 'protein', 2: 'chemical'}

EMBEDDING_DIMS = {
    'dna': 12,
    'protein': 4,
    'chemical': 6,
}
MAX_SEQ_DIM = max(EMBEDDING_DIMS.values())


MODE_TO_ID = {
    'crispri': 0,
    'crispra': 1,
    'overexpression': 2,
    'knockout': 3,
    'inhibitor': 4,
    'agonist': 5,
    'degrader': 6,
    'binder': 7,
    'unknown': 8
}

We'll also build a simple function to make it easy to create data. 

In [4]:
def mock_data(dim_1, dim_2, low=1, high=9):
    return torch.from_numpy(np.round(np.random.uniform(low, high, size=(dim_1, dim_2)), 0)).float()

**Cell 1, Perturbation 1** - A CRISPRa perturbation where we just know the target protein (AA).

You'll notice we create a zero-vector for the sequence and use `has_seq=False` to flag that it's not valid. The modality is still set to DNA for simplicity but it will be ignored as you'll see. 

In [5]:
seq_1 = torch.zeros(1, MAX_SEQ_DIM).squeeze(0)  # no sequence available
tgt_1 = mock_data(1, EMBEDDING_DIMS['protein']).squeeze(0)   # target protein's AA embedding
mod_1 = MODALITY_TO_ID['dna']
mode_1 = MODE_TO_ID['crispra']          
has_seq_1 = False
has_tgt_1 = True

seq_1, tgt_1, mod_1, mode_1

(tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 tensor([3., 2., 3., 5.]),
 0,
 1)

**Cell 1, Perturbation 2** - A CRISPRi perturbation where we know the sgRNA sequence (DNA) and the target protein encoding gene (AA).

You'll see both seq and tgt are set to valid values. 

In [6]:
seq_2 = mock_data(1, EMBEDDING_DIMS['dna']).squeeze(0)   # sgRNA embedding
tgt_2 = mock_data(1, EMBEDDING_DIMS['protein']).squeeze(0)   # gene's protein AA embedding
mod_2 = MODALITY_TO_ID['dna']
mode_2 = MODE_TO_ID['crispri']          
has_seq_2 = True
has_tgt_2 = True

seq_2, tgt_2, mod_2, mode_2

(tensor([4., 5., 3., 9., 7., 2., 4., 6., 2., 9., 5., 7.]),
 tensor([7., 4., 4., 6.]),
 0,
 0)

**Cell 2, Perturbation 1** - A drug based perturbation where we know the chemical sequence (SMILES) and the target protein (AA).

Since the chemical embedding is shorter than the DNA embedding, we pad it with zeros to the shared maximum sequence dimension. 

In [7]:
seq_3_raw = mock_data(1, EMBEDDING_DIMS['chemical'])   # SMILES embedding
seq_3 = torch.zeros(1, MAX_SEQ_DIM) # padded tensor
seq_3[:, :EMBEDDING_DIMS['chemical']] = seq_3_raw
seq_3 = seq_3.squeeze(0)

tgt_3 = mock_data(1, EMBEDDING_DIMS['protein']).squeeze(0)   # gene's protein AA embedding
mod_3 = MODALITY_TO_ID['chemical']
mode_3 = MODE_TO_ID['inhibitor']          
has_seq_3 = True
has_tgt_3 = True

seq_3, tgt_3, mod_3, mode_3

(tensor([7., 3., 3., 6., 5., 2., 0., 0., 0., 0., 0., 0.]),
 tensor([4., 3., 5., 8.]),
 2,
 4)

**Cell 2, Padding** Since our example supports 2 perturbation slots per cell and cell 2 only has 1 real perturbation, we create a padding slot. We'll use a flag later on to tell the model to skip this one entirely, so the values are arbitrary.

In [8]:
seq_4 = torch.zeros(1, MAX_SEQ_DIM).squeeze(0)
tgt_4 = torch.zeros(1, EMBEDDING_DIMS['protein']).squeeze(0)
mod_4, mode_4 = 0, 0
has_seq_4, has_tgt_4 = False, False

seq_4, tgt_4, mod_4, mode_4

(tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 tensor([0., 0., 0., 0.]),
 0,
 0)

### Create Combined Dataset

Now we'll stack the individual perturbation representations into the action composer's input tensors. The first dimension is the batch (cells), the second is the perturbation slot, and the third is the bioFM based embedding of the perturbation. A couple things to notice:
1. For sequence embedding, the first cell only has the second perturbation slot filled while the second cell only has the first slot filled
2. We have target embeddings for all three real perturbations.  This isn't necessary, but you must have at minimum either a sequence or target to be considered a perturbation. 

In [9]:
seq_emb = torch.stack([
    torch.stack([seq_1, seq_2], dim=0), 
    torch.stack([seq_3, seq_4], dim=0)
], dim=0)
seq_emb.shape, seq_emb

(torch.Size([2, 2, 12]),
 tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [4., 5., 3., 9., 7., 2., 4., 6., 2., 9., 5., 7.]],
 
         [[7., 3., 3., 6., 5., 2., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]]))

In [10]:
target_emb = torch.stack([
    torch.stack([tgt_1, tgt_2], dim=0), 
    torch.stack([tgt_3, tgt_4], dim=0)
], dim=0)
target_emb.shape, target_emb

(torch.Size([2, 2, 4]),
 tensor([[[3., 2., 3., 5.],
          [7., 4., 4., 6.]],
 
         [[4., 3., 5., 8.],
          [0., 0., 0., 0.]]]))

In [11]:
modality_ids = torch.tensor([[mod_1, mod_2], [mod_3, mod_4]])
modality_ids.shape, modality_ids

(torch.Size([2, 2]),
 tensor([[0, 0],
         [2, 0]]))

In [12]:
mode_ids = torch.tensor([[mode_1, mode_2], [mode_3, mode_4]])
mode_ids.shape, mode_ids

(torch.Size([2, 2]),
 tensor([[1, 0],
         [4, 0]]))

**Validity Flags** 

To process efficiently in batches, we use a series of validity flags to represent 
1. Which perturbations have valid sequences `has_seq`
2. Which perturbations have valid targets `has_target`
3. Which perturbations are valid `pert_mask`

These flags, as you'll see later, are used to do batch operations more efficiently. 

In [13]:
has_seq = torch.tensor([[has_seq_1, has_seq_2], [has_seq_3, has_seq_4]])
has_target = torch.tensor([[has_tgt_1, has_tgt_2], [has_tgt_3, has_tgt_4]])

has_seq.shape, has_seq, has_target.shape, has_target

(torch.Size([2, 2]),
 tensor([[False,  True],
         [ True, False]]),
 torch.Size([2, 2]),
 tensor([[ True,  True],
         [ True, False]]))

In [14]:
pert_mask = torch.tensor([[True, True], [True, False]])
pert_mask

tensor([[ True,  True],
        [ True, False]])

## Forward Pass

We now have our inputs: `seq_emb`, `target_emb`, `modality_ids`, `mode_ids`, `has_seq`, `has_target`, `pert_mask`.

We're now ready to walk through the forward pass that shows how our model creates a unifying representation from different levels of information.  To unify the representations, we use linear encoding to project each perturbation embedding into the same dimensions space, fusion to pull together perturbations with both sequence and target, and then FiLM to pull the impact of the mode and the perturbation together. The value of the FiLM projections is that we can use the mode to both scale the perturbation embedding multiplicatively and additively giving it a non-linear based scaling.  The result is we apply the following:

$$\text{action}_p = (\text{s}_{latent} + \text{t}_{latent}) \odot (1 + \gamma_\text{mode}) + \beta_\text{mode}$$

- $\text{s}_{latent} = W_{\text{mode}}\cdot \text{e}_{\text{seq}} + b$ is the modality-specific sequence projection (DNA, Protein, Chemical)
- $\text{t}_{latent} = W_\text{target}\cdot \text{e}_{\text{target}} + b$ is the target protein projection
- $\gamma_\text{mode} = W_\gamma\cdot\text{E}_{mode\_embedding}$
- $\beta_\text{mode} = W_\beta\cdot\text{E}_{mode\_embedding}$


We'll start by configuring the unifying embedding dimensions. The `latent_dim` is the shared space all perturbation types are projected into, and `mode_dim` is the size of the mode embedding that controls FiLM conditioning. The composer iterates over each perturbation slot across the whole batch, meaning it will look first at the first perturbation for each cell example, then the second, and so on until it processes all of them. 

In [15]:
B, N_pert = modality_ids.shape
latent_dim = 5
mode_dim = 3

B, N_pert

(2, 2)

**Output preallocation** 

We preallocate our output tensor as zeros. This serves two purposes: reserving memory, and seamlessly handling padding. The forward pass only writes to slots where `pert_mask=True`, so padding slots remain as zeros without any additional logic. 

*Note that you'll also see we add in re-masking at the end of each perturbation to ensure masking integrity*

In [16]:
action_latents = torch.zeros(B, N_pert, latent_dim)
action_latents.shape, action_latents

(torch.Size([2, 2, 5]),
 tensor([[[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]],
 
         [[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]]]))

### Perturbation Slot $p=0$

The composer processes one perturbation slot at a time across the entire batch. For each slot, it loops over the three modality types (DNA, protein, chemical) and uses a boolean mask to route each batch row through its correct linear projector.

We'll walk through slot $p=0$ step by step. In our example, this is cell 1's CRISPRa (no sequence, target only) perturbation and cell 2's chemical inhibitor (sequence + target) perturbation.   

In [17]:
p = 0

**Perturbation validity** 

We first check to see which batch rows have a real perturbation in this slot. If the entire column were `False`, the loop would skip to the next slot (function not shown)

In [18]:
p_mask = pert_mask[:, p]
p_mask.shape, p_mask

(torch.Size([2]), tensor([True, True]))

**Extract inputs** 

Next we slice out the current perturbation slot's metadata for all batch rows. When we slice we also ensure that padding slots don't trigger sequence or target encoding. Since this is the first slot, you'll see that we only have a sequence for the second cell, but a target for both. 

In [19]:
p_has_seq = has_seq[:, p] & p_mask
p_has_target = has_target[:, p] & p_mask
p_modality = modality_ids[:, p]
p_mode = mode_ids[:, p]
p_has_seq, p_has_target, p_modality, p_mode

(tensor([False,  True]), tensor([True, True]), tensor([0, 2]), tensor([1, 4]))

#### Sequence Encoding Initialization

We'll now encode the sequences. The composer uses separate linear projectors for each modality type to project from the modality dimension to our target dimension resulting in:

$$\mathbf{s}_p = W_{\text{modality}} \cdot \text{embedding}_\text{seq} + b$$

We have 3 different weights, one for each modality, allowing the model to learn how to best project each embedding to the unified layer.  We'll start by initializing the 3 layers. 

*Note that in the actual model you'll see the following code written in loops to collapse iterations better*

**Modality Weights**

We'll now create the three modality-specific linear projectors. Each one takes the modality's FM embedding dimension and projects to the shared dimension `latent_dim`. 

To show this projection, we'll use 3 different consistent weight initialization to highlight how the weights can diverge. We'll initialize
1. DNA based projection (dna_layer)
2. Protein based projection (prot_layer)
3. Chemical based projection (chem_layer)

In [20]:
dna_layer = nn.Linear(EMBEDDING_DIMS['dna'], latent_dim)
nn.init.constant_(dna_layer.weight, 1.5)
nn.init.zeros_(dna_layer.bias)
dna_layer.weight

Parameter containing:
tensor([[1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
         1.5000, 1.5000, 1.5000],
        [1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
         1.5000, 1.5000, 1.5000],
        [1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
         1.5000, 1.5000, 1.5000],
        [1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
         1.5000, 1.5000, 1.5000],
        [1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
         1.5000, 1.5000, 1.5000]], requires_grad=True)

In [21]:
prot_layer = nn.Linear(EMBEDDING_DIMS['protein'], latent_dim)
nn.init.constant_(prot_layer.weight, 0.5)
nn.init.zeros_(prot_layer.bias)
prot_layer.weight

Parameter containing:
tensor([[0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000]], requires_grad=True)

In [22]:
chem_layer = nn.Linear(EMBEDDING_DIMS['chemical'], latent_dim)
nn.init.constant_(chem_layer.weight, -1.0)
nn.init.zeros_(chem_layer.bias)
chem_layer.weight

Parameter containing:
tensor([[-1., -1., -1., -1., -1., -1.],
        [-1., -1., -1., -1., -1., -1.],
        [-1., -1., -1., -1., -1., -1.],
        [-1., -1., -1., -1., -1., -1.],
        [-1., -1., -1., -1., -1., -1.]], requires_grad=True)

**Unify** 

Now that we have them initialized, we want to simplify how we'll use them.  We'll create a module dictionary so that we can easily index out the linear projection we want based on the modality name. 

In [23]:
seq_projectors = nn.ModuleDict({
    'dna': dna_layer,
    'protein': prot_layer,
    'chemical': chem_layer,
})
seq_projectors

ModuleDict(
  (dna): Linear(in_features=12, out_features=5, bias=True)
  (protein): Linear(in_features=4, out_features=5, bias=True)
  (chemical): Linear(in_features=6, out_features=5, bias=True)
)

#### Sequence Encoding by Modality 
Now we're ready to actually run our sequence encoding.  In practice we'd loop through the modalities.  Here we'll write it out one by one.  Since we know the modalities we have, we'll take a shortcut on some of the loops (we do this using an `if ... continue` loop in code). You'll see that for this first layer, we only have a chemical sequence, so we'll quickly cycle through the first two modalities and jump to the third. 

We'll first start by again creating the placeholder output to simplify masking and extract our slice of `has_seq`

In [24]:
seq_lat = torch.zeros(B, latent_dim)
seq_lat.shape, seq_lat

(torch.Size([2, 5]),
 tensor([[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]))

**Modality routing: DNA (mod_id=0)** 

We'll run our first check to see if any valid perturbations have a modality of 0.  This requires both checking the modality slice `p_modality` and that that slice is a sequence `p_has_seq`. We have to check both since you'll see that the first index has a `0` modality, but we have flagged it as a padding, not a valid entry. Because of this dual check, we can skip to our next modality. 

In [25]:
mod_id = 0 
p_has_modality = p_modality == mod_id
mod_mask = p_has_modality & p_has_seq
p_has_modality, mod_mask

(tensor([ True, False]), tensor([False, False]))

**Modality routing: Protein (mod_id=1)** 

Now we'll check if any of our perturbations are proteins.  We'll quickly see they're not and jump to our next modality. 

In [26]:
mod_id = 1
p_has_modality = p_modality == mod_id
mod_mask = p_has_modality & p_has_seq
p_has_modality, mod_mask

(tensor([False, False]), tensor([False, False]))

**Modality routing: Chemical (mod_id=2)** 

Finally we'll check for chemical perturbations.  This is the one we expect to see for our second cell. We'll see that our checks finally overlap showing us a valid position to process. 

In [27]:
mod_id = 2
p_has_modality = p_modality == mod_id
mod_mask = p_has_modality & p_has_seq
p_has_modality, mod_mask

(tensor([False,  True]), tensor([False,  True]))

**Extract projector**  

Now that we know we have a chemical based perturbation, we'll extract the chemical linear projector.  Here you'll see where our dictionary and code dynamically extracts the right projector from our dictionary of sequence projectors. 

In [28]:
mod_id_key = ID_TO_MODALITY[mod_id]
proj = seq_projectors[mod_id_key]
mod_id_key, proj 

('chemical', Linear(in_features=6, out_features=5, bias=True))

**Project sequence** 

Now that we have the projector and know which sequences are expected to use it, we're ready to do our projection. We'll use our flags to index out the perturbations that apply to this projection and then run them through our linear layer. Because we initialized our weights to -1 across all embedding dimensions, you'll see our linear projection is just the negative sum of the row for each value. 

*Notice that we also cap the length of our embedding input during this projection. This ensures we don't pass in a longer embedding than our projection can handle. This does raise the risk of silent failures for dimension mismatches but we're accepting that risk*

In [29]:
seq_to_project = seq_emb[mod_mask, p, :proj.in_features]
seq_lat[mod_mask] = proj(seq_to_project)
seq_to_project, seq_lat.shape, seq_lat

(tensor([[7., 3., 3., 6., 5., 2.]]),
 torch.Size([2, 5]),
 tensor([[  0.,   0.,   0.,   0.,   0.],
         [-26., -26., -26., -26., -26.]], grad_fn=<IndexPutBackward0>))

#### Target Encoding

Now that we have sequences encoded, we encode the targets. At this point, our action composer only supports protein encoded targets so all the dimensions are the same. If we wanted to support other modalities of targets, we'd have to create a similar projector like we did for sequences. For our target, we again use a linear projector:

$$\mathbf{t}_p = W_\text{target} \cdot \text{embedding}_\text{target} + b$$

Since not all perturbations require targets, we'll similarly initialize our target latent and then fill in the entries that are valid. 

In [30]:
target_lat = torch.zeros(B, latent_dim)
target_lat.shape, target_lat

(torch.Size([2, 5]),
 tensor([[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]))

**Target Weights**

We'll now create the weights for our target.  You'll see that they project from the protein embedding dimension to our same latent dimension, creating the same size tensor as our sequence embedding output.  This will allow us to fuse the target and sequence together. 

In [31]:
target_projector = nn.Linear(EMBEDDING_DIMS['protein'], latent_dim)
nn.init.constant_(target_projector.weight, 0.25)
nn.init.zeros_(target_projector.bias)
target_projector.weight

Parameter containing:
tensor([[0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500]], requires_grad=True)

**Project target** 

Now we're ready to project our target. Similar to sequence projecting, we use the target flag to only project the valid targets and then save them back to our target latent. Both of our perturbations in this batch have a target sequence, so we'll see that we output latents for both perturbations in the batch.   Additionally since we initialized across all dimensions with the same value, the projection will be the same for each embedding channel for a sample. 

*Notice that in this case we do not provide a safety cap on the target sequence input. This is because we're more concerned if the sequence doesn't fit the expected dimension since we expect all targets to be the same dimension*

In [32]:
targ_seq = target_emb[p_has_target, p]
target_lat[p_has_target] = target_projector(targ_seq)
targ_seq, target_lat.shape, target_lat

(tensor([[3., 2., 3., 5.],
         [4., 3., 5., 8.]]),
 torch.Size([2, 5]),
 tensor([[3.2500, 3.2500, 3.2500, 3.2500, 3.2500],
         [5.0000, 5.0000, 5.0000, 5.0000, 5.0000]], grad_fn=<IndexPutBackward0>))

#### Additive Fusion

Now that we have our sequence and target latents for the first perturbation across all batches, we're ready to create a single representation of the perturbation. We do this through fusion, or simply adding the two latents together.   This step allows the model to support perturbations that have either a sequence, a target, or both.   The fusion is represented as follows:
$$c = (\text{s}_{latent} + \text{t}_{latent})$$

Before running fusion we add in dropout. The dropout layer we run does two things: 
1. Independently, item by item, does zeros elements based on sampling from a Bernoulli distribution.  This means that even if we set the dropout to 0.1 and have 10 cells, we do not have a guarantee that 1 cell will be set to 0. Recall that with independent sampling, the probability that 10 cells all have values if we have a 10% chance of zeroing is $(1 - p)^{n} = 0.9^{10}$ which is ~ 34%. This zeroing prevents the model from relying too heavily on any specific value in either the sequence or the target which is helpful given our datasets flexibility on what data is provided for a perturbation.
2. All values are normalized to account for the addition of zeroed cells using $\frac{1}{1-p}$.  This keeps the expected value of the output unchanged between training that runs with dropout and inference that runs without dropout ensuring we don't need to rescale at test time.  

*In the production code you'll also see the model also includes a learned `unknown_embedding` for cases where neither sequence nor target is available, but our current data pipeline always provides at least one so it never fires in practice.*

**Initiate Dropout** 

The dropout layer is a non-learnable layer but has a great benefit that it only calculates when the model is in training mode.  If the model is flipped to eval, this layer is skipped. We'll start and initialize with a probability of 0.4 to push the probability high enough that we see it fire on our dataset. 

In [33]:
latent_dropout = nn.Dropout(p=0.4)
latent_dropout

Dropout(p=0.4, inplace=False)

**Dropout sequence**

We'll now run our dropout on the sequence. You'll see that we both create some zero values and we scale the remaining values.  Since the whole first entry is zeros, it's hard to know just how many of that entry were "zeroed" and scaled. 

In [34]:
seq_masked = latent_dropout(seq_lat)
seq_masked

tensor([[  0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
        [ -0.0000, -43.3333,  -0.0000,  -0.0000,  -0.0000]],
       grad_fn=<MulBackward0>)

**Dropout target** 

Now we'll run dropout on the targets.  Again, you'll see some zero values added while all the values are scaled. 

In [35]:
target_masked = latent_dropout(target_lat)
target_masked

tensor([[5.4167, 5.4167, 5.4167, 5.4167, 5.4167],
        [8.3333, 8.3333, 8.3333, 0.0000, 8.3333]], grad_fn=<MulBackward0>)

**Fusion** 

Now we'll run our fusion. This is simply a sum of the two vectors.  For this example, since we have the first cell example that only has a target, and the second example that has a sequence and target, you'll get to see how fusion supports both examples to create an output where both have non-zero embeddings.  Over time, the weights on the linear projections will adjust as the model learns how and when to prioritize the different sequence and target embeddings. 

In [36]:
content = seq_masked + target_masked
content.shape, content

(torch.Size([2, 5]),
 tensor([[  5.4167,   5.4167,   5.4167,   5.4167,   5.4167],
         [  8.3333, -35.0000,   8.3333,   0.0000,   8.3333]],
        grad_fn=<AddBackward0>))

#### FiLM Mode Conditioning

Now that we have our perturbation encoded, we're ready to include the mode to scale the encoding in the right direction and scale. We use a Feature-wise Linear Modulation (FiLM) approach as it lets the model learn how to shift the perturbation embedding with a multiplier and a sum unit.  The FiLM calculation is:

$$\mathbf{a} = \mathbf{c} \odot (1 + \gamma_\text{mode}) + \beta_\text{mode}$$

Both the scale ($\gamma$) and shift ($\beta$) have learnable weights as follows: 

- $\gamma_\text{mode} = W_\gamma\cdot\text{E}_{mode\_embedding}$
- $\beta_\text{mode} = W_\beta\cdot\text{E}_{mode\_embedding}$

We'll first do light data cleanup to ensure we don't have invalid modes and create learnable embeddings for our mode. We'll start with a clamp function to remove any invalid modes. 

In [37]:
num_modes = max(MODE_TO_ID.values())
p_mode = p_mode.clamp(0, num_modes)
num_modes, p_mode

(8, tensor([1, 4]))

**Mode embedding** 

Now we'll create a mode embedding layer.  To drive different mode values, we'll initialize our embedding with a row-incremental weight. 

In [38]:
mode_embedding = nn.Embedding(num_modes, mode_dim)
vs, d = mode_dim, num_modes
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 0.1*cols)  
mode_embedding.weight = nn.Parameter(pattern)
mode_embedding.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000],
        [0.7000, 0.7000, 0.7000],
        [0.8000, 0.8000, 0.8000]], requires_grad=True)

**Mode Embedding Projection** 

Now we're ready to do an embedding projection of each row. This will in essence pluck out the row for each embedding, so we'll see the second and fifth rows pulled out. 

In [39]:
mode_vecs = mode_embedding(p_mode)
mode_vecs.shape, mode_vecs

(torch.Size([2, 3]),
 tensor([[0.2000, 0.2000, 0.2000],
         [0.5000, 0.5000, 0.5000]], grad_fn=<EmbeddingBackward0>))

**FiLM scale $\gamma_\text{mode}$**

Now that we have our embedding, we can calculate the scale component of our FiLM calculation. This will be multiplied against the perturbation encoding to produce a hadamard product meaning we'll have element wise gradients that this layer can learn from.  We'll initialize this statically so you'll see that we end up with uniform values by example. 

In [40]:
film_scale = nn.Linear(mode_dim, latent_dim)
nn.init.constant_(film_scale.weight, 0.1)
nn.init.zeros_(film_scale.bias)
film_scale.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000],
        [0.1000, 0.1000, 0.1000],
        [0.1000, 0.1000, 0.1000],
        [0.1000, 0.1000, 0.1000],
        [0.1000, 0.1000, 0.1000]], requires_grad=True)

In [41]:
scale = film_scale(mode_vecs)
scale.shape, scale

(torch.Size([2, 5]),
 tensor([[0.0600, 0.0600, 0.0600, 0.0600, 0.0600],
         [0.1500, 0.1500, 0.1500, 0.1500, 0.1500]], grad_fn=<AddmmBackward0>))

**FiLM shift $\beta_\text{mode}$**

Now we'll calculate the shift component.  This is added to the product of the embedding and scale, acting almost like a larger bias term.  We'll initialize this to a consistent negative weight to show that it can be used to scale down values. 

In [42]:
film_shift = nn.Linear(mode_dim, latent_dim)
nn.init.constant_(film_shift.weight, -0.2)
nn.init.zeros_(film_shift.bias)
film_shift.weight

Parameter containing:
tensor([[-0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000]], requires_grad=True)

In [43]:
shift = film_shift(mode_vecs)
shift.shape, shift

(torch.Size([2, 5]),
 tensor([[-0.1200, -0.1200, -0.1200, -0.1200, -0.1200],
         [-0.3000, -0.3000, -0.3000, -0.3000, -0.3000]],
        grad_fn=<AddmmBackward0>))

**FiLM calculation**

Now we're ready for the final calculation of the perturbation that takes into account the perturbation sequence, target, and the mode.  You'll see that this results in a complex output highlighting the value of the different layers we have. 

In [44]:
action = content * (1.0 + scale) + shift
action.shape, action

(torch.Size([2, 5]),
 tensor([[  5.6217,   5.6217,   5.6217,   5.6217,   5.6217],
         [  9.2833, -40.5500,   9.2833,  -0.3000,   9.2833]],
        grad_fn=<AddBackward0>))

#### Save perturbation to batch
Recall that we've just calculated the first perturbation for each cell.  We also pre-allocated the output in `action_latents`. We now need to save our action to the first index of each example.  Before we do, though, we'll do one final re-masking just in case we had an error or leakage happen. Since our mask is a boolean we can do multiplication to zero out anything that needs to be masked, then we'll write to the first slice of entries in the action_latent. 

In [45]:
action = action * p_mask.float().unsqueeze(-1)
action.shape, action

(torch.Size([2, 5]),
 tensor([[  5.6217,   5.6217,   5.6217,   5.6217,   5.6217],
         [  9.2833, -40.5500,   9.2833,  -0.3000,   9.2833]],
        grad_fn=<MulBackward0>))

In [46]:
action_latents[:, p] = action
action_latents.shape, action_latents

(torch.Size([2, 2, 5]),
 tensor([[[  5.6217,   5.6217,   5.6217,   5.6217,   5.6217],
          [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000]],
 
         [[  9.2833, -40.5500,   9.2833,  -0.3000,   9.2833],
          [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000]]],
        grad_fn=<CopySlices>))

### Perturbation Slot $p=1$

Since we're linearly going through our perturbations instead of a loop, we now have to go on to our next perturbation.  Luckily our example only has a max of 2 so we'll just go back through the steps a second time.  Recall that for the second perturbation, our first cell has a CRISPRi perturbation that has sequence and target embeddings available and our second cell has padding as there is no second perturbation. 

*For this pass through the loop, I'll mainly highlight differences that you'll observe.  We'll also skip our layer initializations as they've already been completed*

We'll start by incrementing our position. 

In [47]:
p = 1

**Perturbation validity** 

This time you'll see that only the first example is valid. You'll see that this will impact our downstream analysis. 

In [48]:
p_mask = pert_mask[:, p]
p_mask.shape, p_mask

(torch.Size([2]), tensor([ True, False]))

**Extract inputs** 

You'll see that our input extraction is now impacted where the second cell is invalid and set to false. This shows the power of including both the mask and indexing.  

In [49]:
p_has_seq = has_seq[:, p] & p_mask
p_has_target = has_target[:, p] & p_mask
p_modality = modality_ids[:, p]
p_mode = mode_ids[:, p]
p_has_seq, p_has_target, p_modality, p_mode

(tensor([ True, False]),
 tensor([ True, False]),
 tensor([0, 0]),
 tensor([0, 0]))

#### Sequence Encoding by Modality 

In [50]:
seq_lat = torch.zeros(B, latent_dim)
seq_lat.shape, seq_lat

(torch.Size([2, 5]),
 tensor([[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]))

**Modality routing: DNA (mod_id=0)** 

This time the first cell's CRISPRi perturbation triggers our first modality. You'll see the overlap between our two flags. 

In [51]:
mod_id = 0 
p_has_modality = p_modality == mod_id
mod_mask = p_has_modality & p_has_seq
p_has_modality, mod_mask

(tensor([True, True]), tensor([ True, False]))

**Extract projector** 

Since we're on the first modality, we'll need to pull out the linear projection for DNA based sequences.  Luckily we've written our code to be dynamic and so the code looks the exact same as the chemical projection. 

In [52]:
mod_id_key = ID_TO_MODALITY[mod_id]
proj = seq_projectors[mod_id_key]
mod_id_key, proj

('dna', Linear(in_features=12, out_features=5, bias=True))

**Project** 

You'll see that we have quite a bit larger input embedding for DNA, but because our linear layers all output the same dimensions, we see the same size output for this modality as we saw with others. 

In [53]:
seq_to_project = seq_emb[mod_mask, p, :proj.in_features]
seq_lat[mod_mask] = proj(seq_to_project)
seq_to_project, seq_lat.shape, seq_lat

(tensor([[4., 5., 3., 9., 7., 2., 4., 6., 2., 9., 5., 7.]]),
 torch.Size([2, 5]),
 tensor([[94.5000, 94.5000, 94.5000, 94.5000, 94.5000],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000]],
        grad_fn=<IndexPutBackward0>))

**Modality routing: Protein (mod_id=1)** 

We're ready to check the other modalities.  You'll see that the remaining two modalities both come up with no valid entries, as expected. 

In [54]:
mod_id = 1
p_has_modality = p_modality == mod_id
mod_mask = p_has_modality & p_has_seq
p_has_modality, mod_mask

(tensor([False, False]), tensor([False, False]))

**Modality routing: Chemical (mod_id=2)** 

In [55]:
mod_id = 2
p_has_modality = p_modality == mod_id
mod_mask = p_has_modality & p_has_seq
p_has_modality, mod_mask

(tensor([False, False]), tensor([False, False]))

#### Target Encoding

The target encoding will flip this time.  We'll see that example 1 has a target, but example 2 does not.  We'll follow the same steps of preallocating and projecting the target. 

In [56]:
target_lat = torch.zeros(B, latent_dim)
target_lat.shape, target_lat

(torch.Size([2, 5]),
 tensor([[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]))

**Project target** 

In [57]:
targ_seq = target_emb[p_has_target, p]
target_lat[p_has_target] = target_projector(targ_seq)
targ_seq, target_lat.shape, target_lat

(tensor([[7., 4., 4., 6.]]),
 torch.Size([2, 5]),
 tensor([[5.2500, 5.2500, 5.2500, 5.2500, 5.2500],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000]], grad_fn=<IndexPutBackward0>))

#### Additive Fusion

We'll follow the same fusion process with dropout. Since the second cell is not valid though, you'll see the fusion process still results in all zeros. 

**Dropout sequence** 

In [58]:
seq_masked = latent_dropout(seq_lat)
seq_masked

tensor([[  0.0000, 157.5000,   0.0000, 157.5000,   0.0000],
        [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000]],
       grad_fn=<MulBackward0>)

**Dropout target**

In [59]:
target_masked = latent_dropout(target_lat)
target_masked

tensor([[8.7500, 0.0000, 8.7500, 0.0000, 8.7500],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000]], grad_fn=<MulBackward0>)

**Fusion** 

In [60]:
content = seq_masked + target_masked
content.shape, content

(torch.Size([2, 5]),
 tensor([[  8.7500, 157.5000,   8.7500, 157.5000,   8.7500],
         [  0.0000,   0.0000,   0.0000,   0.0000,   0.0000]],
        grad_fn=<AddBackward0>))

#### FiLM Mode Conditioning

We'll follow the same steps for our FiLM scale and shift.  You'll notice that with our FiLM calculations we don't actually take in a flag, so we calculate the scale and shift for all examples in the batch and then remove them after our FiLM calculation is complete. 

In [61]:
num_modes = max(MODE_TO_ID.values())
p_mode = p_mode.clamp(0, num_modes)
num_modes, p_mode

(8, tensor([0, 0]))

**Mode Embedding Projection** 

In [62]:
mode_vecs = mode_embedding(p_mode)
mode_vecs.shape, mode_vecs

(torch.Size([2, 3]),
 tensor([[0.1000, 0.1000, 0.1000],
         [0.1000, 0.1000, 0.1000]], grad_fn=<EmbeddingBackward0>))

**FiLM scale $\gamma_\text{mode}$**

In [63]:
scale = film_scale(mode_vecs)
scale.shape, scale

(torch.Size([2, 5]),
 tensor([[0.0300, 0.0300, 0.0300, 0.0300, 0.0300],
         [0.0300, 0.0300, 0.0300, 0.0300, 0.0300]], grad_fn=<AddmmBackward0>))

**FiLM shift $\beta_\text{mode}$**

In [64]:
shift = film_shift(mode_vecs)
shift.shape, shift

(torch.Size([2, 5]),
 tensor([[-0.0600, -0.0600, -0.0600, -0.0600, -0.0600],
         [-0.0600, -0.0600, -0.0600, -0.0600, -0.0600]],
        grad_fn=<AddmmBackward0>))

**FiLM calculation**

You'll notice that our second example is non-zero here because of our FiLM scale and shift.  Don't worry, we'll clean that up soon.

In [65]:
action = content * (1.0 + scale) + shift
action.shape, action

(torch.Size([2, 5]),
 tensor([[ 8.9525e+00,  1.6216e+02,  8.9525e+00,  1.6216e+02,  8.9525e+00],
         [-6.0000e-02, -6.0000e-02, -6.0000e-02, -6.0000e-02, -6.0000e-02]],
        grad_fn=<AddBackward0>))

#### Save perturbation to batch

Now we'll do our quick cleanup and zero out the second example using our `p_mask`.  This will help remove the calculated perturbation so that we can write to our output. 

In [66]:
action = action * p_mask.float().unsqueeze(-1)
action.shape, action

(torch.Size([2, 5]),
 tensor([[  8.9525, 162.1650,   8.9525, 162.1650,   8.9525],
         [ -0.0000,  -0.0000,  -0.0000,  -0.0000,  -0.0000]],
        grad_fn=<MulBackward0>))

In [67]:
action_latents[:, p] = action
action_latents.shape, action_latents

(torch.Size([2, 2, 5]),
 tensor([[[  5.6217,   5.6217,   5.6217,   5.6217,   5.6217],
          [  8.9525, 162.1650,   8.9525, 162.1650,   8.9525]],
 
         [[  9.2833, -40.5500,   9.2833,  -0.3000,   9.2833],
          [ -0.0000,  -0.0000,  -0.0000,  -0.0000,  -0.0000]]],
        grad_fn=<CopySlices>))

## Action Latent Complete

Now that we've processed every perturbation slot, we have a full perturbation latent representation. 

In [68]:
action_latents.shape, action_latents

(torch.Size([2, 2, 5]),
 tensor([[[  5.6217,   5.6217,   5.6217,   5.6217,   5.6217],
          [  8.9525, 162.1650,   8.9525, 162.1650,   8.9525]],
 
         [[  9.2833, -40.5500,   9.2833,  -0.3000,   9.2833],
          [ -0.0000,  -0.0000,  -0.0000,  -0.0000,  -0.0000]]],
        grad_fn=<CopySlices>))

# Latent Representation

We now have a latent representation of all the perturbations by cell for the batch. You'll notice that we've unified different types of perturbations (DNA, chemical, target-only) and different modes (CRISPRa, CRISPRi, inhibitor) into a common $[B, N_{\text{pert}}, D]$ tensor and ensured we've removed representation of any non-perturbation slots. 

This is now in a structure that's ready to pass to the Action Conditioned Predictor or to evaluations on the composer.